# 01 — Review, approve, and execute a PhysioNet contract

This notebook connects assisted authoring to deterministic execution. Cohere may translate an explicit researcher declaration into the shared payload schema, but it cannot create approval metadata or run the study.

The candidate must match the declared payload, pass local checks, and receive explicit researcher approval before FeatureGraph can execute it.

## 1. Configure the review boundary

Leave `RESEARCHER_APPROVES = False` until you have inspected the candidate, differences, and validation table. The API key is read from the environment and is never printed.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import tempfile
from copy import deepcopy
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import JSON, display
from jsonschema import Draft202012Validator

import featuregraph as fg
from scripts.run_physionet_wearable_protocol_study import (
    EXCLUSIONS,
    SELF_REPORT_COLUMNS,
    SIGNAL_FILES,
    SUBJECTS,
    cohort_for,
    run_study,
)

RUN_COHERE = False
RESEARCHER_APPROVES = False
RESEARCHER_AUTHORITY = "Nazia Habib"
COHERE_MODEL = "command-a-plus-05-2026"
COHERE_API_KEY = os.environ.get("COHERE_API_KEY")


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the FeatureGraph repository."
    )


REPO_ROOT = find_repo_root()
STUDY_DIR = REPO_ROOT / "artifacts" / "studies" / "physionet_wearable"
APPROVED_CONTRACT_PATH = STUDY_DIR / "study_contract.json"
PAYLOAD_SCHEMA_PATH = STUDY_DIR / "study_contract_payload.schema.json"
print(f"Repository: {REPO_ROOT}")
print(f"Cohere requests enabled: {RUN_COHERE}")
print(f"API key available: {bool(COHERE_API_KEY)}")

## 2. Load the shared payload schema and maintained reference

Approval metadata is removed before authoring and comparison. The candidate and the deterministic runner therefore operate on the same researcher-controlled payload.

In [ ]:
payload_schema = json.loads(PAYLOAD_SCHEMA_PATH.read_text(encoding="utf-8"))
Draft202012Validator.check_schema(payload_schema)

maintained_reference = fg.load_approved_study_contract(
    APPROVED_CONTRACT_PATH
)
researcher_declared_payload = fg.study_contract_payload(
    maintained_reference.contract
)
Draft202012Validator(payload_schema).validate(researcher_declared_payload)

print(f"Approved reference: {maintained_reference.sha256}")
display(JSON(researcher_declared_payload))

## 3. Produce a schema-constrained candidate

This first vertical slice gives Cohere a complete researcher declaration and asks it only to preserve that declaration in the canonical payload shape. Any change is exposed by the deterministic diff below. Offline mode uses an exact frozen candidate.

In [ ]:
SYSTEM_MESSAGE = """You serialize a researcher-declared FeatureGraph study payload.
Preserve every supplied value exactly. Do not infer, correct, approve, or execute
anything. Return only one JSON object matching the supplied schema."""
PROMPT_VERSION = "physionet-payload-preservation-v1"
candidate_prompt = (
    "Return this researcher-declared payload without changing any value:\n"
    + json.dumps(researcher_declared_payload, sort_keys=True, indent=2)
)


def call_cohere_candidate(
    prompt: str, schema: dict[str, Any]
) -> tuple[dict[str, Any], dict[str, Any]]:
    if not COHERE_API_KEY:
        raise RuntimeError(
            "Set COHERE_API_KEY before enabling Cohere requests."
        )
    import cohere

    api_schema = {
        key: value
        for key, value in schema.items()
        if key not in {"$schema", "title"}
    }
    response = cohere.ClientV2(api_key=COHERE_API_KEY).chat(
        model=COHERE_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": prompt},
        ],
        response_format={
            "type": "json_object",
            "json_schema": api_schema,
        },
        temperature=0,
    )
    response_text = response.message.content[0].text
    payload = json.loads(response_text)
    Draft202012Validator(schema).validate(payload)
    return payload, {
        "mode": "cohere",
        "model": COHERE_MODEL,
        "prompt_version": PROMPT_VERSION,
        "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
        "response_sha256": hashlib.sha256(
            response_text.encode()
        ).hexdigest(),
        "response_id": getattr(response, "id", None),
    }


if RUN_COHERE:
    candidate_payload, candidate_provenance = call_cohere_candidate(
        candidate_prompt, payload_schema
    )
else:
    candidate_payload = deepcopy(researcher_declared_payload)
    candidate_provenance = {
        "mode": "offline_exact_candidate",
        "prompt_version": PROMPT_VERSION,
    }

display(JSON(candidate_provenance))
display(JSON(candidate_payload))

## 4. Review every difference and validation result

Schema-valid output is not automatically acceptable. In this preservation test, approval requires exact parity with the researcher declaration and no unresolved questions.

In [ ]:
schema_errors = sorted(
    Draft202012Validator(payload_schema).iter_errors(candidate_payload),
    key=lambda error: list(error.path),
)
differences = fg.study_contract_differences(
    candidate_payload, researcher_declared_payload
)
validation_results = {
    "json_schema": not schema_errors,
    "no_unresolved_questions": (
        candidate_payload.get("unresolved_questions") == []
    ),
    "researcher_declared_parity": not differences,
}
validation_table = pd.DataFrame(
    [
        {"check": name, "passed": passed}
        for name, passed in validation_results.items()
    ]
)
display(validation_table)
display(
    pd.DataFrame(differences)
    if differences
    else pd.DataFrame(columns=["path", "candidate", "reference"])
)

## 5. Record explicit approval

Cohere never receives this control. With approval disabled, the candidate remains ineligible for execution.

In [ ]:
candidate_sha256 = fg.study_contract_sha256(candidate_payload)
approved_candidate = None
if RESEARCHER_APPROVES:
    approved_candidate = fg.approve_study_contract(
        candidate_payload,
        authority=RESEARCHER_AUTHORITY,
        validation_results=validation_results,
    )
    print(
        "Approved candidate: "
        + approved_candidate["approval"]["contract_sha256"]
    )
else:
    print(f"Candidate only: {candidate_sha256}")
    print("Not approved; the maintained contract will be executed below.")

## 6. Execute only an approved contract

The network-free fixture contains all 36 declared subject identifiers, applies the three approved exclusions, and reconstructs the protected 33-participant result. If the candidate is not approved, this cell executes the maintained approved reference instead.

In [ ]:
def fixture_tags(count: int) -> pd.DatetimeIndex:
    return pd.date_range(
        "2026-01-01", periods=count, freq="1min", tz="UTC"
    )


def write_signal(
    path: Path, start: pd.Timestamp, sample_count: int, value: float
) -> None:
    rows = [start.isoformat(), "1.0", *([str(value)] * sample_count)]
    path.write_text("\n".join(rows) + "\n", encoding="utf-8")


def write_protected_fixture(cache: Path) -> None:
    payload = researcher_declared_payload
    protocols = payload["protocol_versions"]
    for cohort, columns in SELF_REPORT_COLUMNS.items():
        cohort_subjects = [
            subject for subject in SUBJECTS if cohort_for(subject) == cohort
        ]
        reports = pd.DataFrame(
            {
                source_column: [float(index + 1)] * len(cohort_subjects)
                for index, source_column in enumerate(columns)
            },
            index=cohort_subjects,
        )
        reports.to_csv(cache / f"Stress_Level_{cohort}.csv")

    for subject_id in SUBJECTS:
        if subject_id in EXCLUSIONS:
            continue
        cohort = cohort_for(subject_id)
        occurrences = protocols[cohort]["occurrences"]
        tag_count = max(
            occurrence["end_tag_index"] for occurrence in occurrences
        ) + 1
        tags = fixture_tags(tag_count)
        subject_dir = (
            cache / "Wearable_Dataset" / "STRESS" / subject_id
        )
        subject_dir.mkdir(parents=True)
        pd.Series(tags.astype(str)).to_csv(
            subject_dir / "tags.csv", index=False, header=False
        )
        sample_count = int((tags[-1] - tags[0]).total_seconds()) + 1
        for signal_index, filename in enumerate(
            SIGNAL_FILES.values(), start=1
        ):
            write_signal(
                subject_dir / filename,
                tags[0],
                sample_count,
                float(signal_index),
            )


contract_for_execution = (
    approved_candidate
    if approved_candidate is not None
    else maintained_reference.contract
)
with tempfile.TemporaryDirectory() as temporary_directory:
    temporary_root = Path(temporary_directory)
    contract_path = temporary_root / "approved_contract.json"
    contract_path.write_text(
        json.dumps(contract_for_execution, indent=2) + "\n",
        encoding="utf-8",
    )
    approved_for_execution = fg.load_approved_study_contract(contract_path)
    cache = temporary_root / "cache"
    cache.mkdir()
    write_protected_fixture(cache)
    results = run_study(cache, approved_for_execution)

execution_summary = pd.DataFrame(
    [
        {
            "eligible_participants": int(
                results["study_objects"]["subject_id"].nunique()
            ),
            "declared_occurrences": int(len(results["study_objects"])),
            "compiler_checks": int(
                len(results["compiler_validation"])
            ),
            "all_checks_passed": bool(
                results["validation"]["passed"].all()
            ),
            "contract_sha256": approved_for_execution.sha256,
        }
    ]
)
display(execution_summary)
assert execution_summary.loc[0, "eligible_participants"] == 33
assert execution_summary.loc[0, "declared_occurrences"] == 248
assert execution_summary.loc[0, "compiler_checks"] == 99
assert execution_summary.loc[0, "all_checks_passed"]

## What this establishes

The assisted candidate, researcher-controlled payload, approval fingerprint, and executed payload now share one boundary. Cohere can prepare a candidate, but only deterministic checks and explicit researcher approval make that candidate executable. The same approved payload reproduces 33 eligible participants, 248 declared protocol occurrences, and 99 compiler checks without another Cohere request.